# Experiment 3 — Dense Semantic Representations

The previous experiments used TF-IDF, which represents text mainly through lexical frequency.

In this experiment, we compare TF-IDF with two dense semantic representations:

1. **Word2Vec trained on the project corpus**
2. **Pretrained GloVe embeddings**

The classifiers remain similar to the previous experiments:

- Logistic Regression for binary classification
- One-vs-Rest Logistic Regression for multi-label classification

This makes the comparison focus mainly on the text representation.

*Uses the same data and train/test split as Experiments 1–2 (`src/data.py`). Run notebooks 01 and 02 first: the final comparison reads their results from `results/`.*

In [ ]:
# 0. Make the shared `src` package importable (works from the repo root or from notebooks/)

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
# Experiment 3 setup
# If gensim is not installed, uncomment the next line and run this cell again:
# %pip install gensim -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from gensim.models import Word2Vec
import gensim.downloader as api

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    classification_report,
    ConfusionMatrixDisplay
)

from src.data import RANDOM_STATE, load_all, load_table, save_table
from src.embeddings import tokenize_text, mean_embedding, vectorize_texts, embedding_coverage

MULTILABEL_COLUMNS = ["urgency", "scarcity", "social_proof", "emotional_pressure"]

print("Example tokenization:")
print(tokenize_text("Only 3 seats left — book now before midnight!"))

In [ ]:
# Load data and the shared split (same as Experiments 1–2)

main_df, groups, real_df, external_synthetic_df, train_idx, test_idx = load_all()

X = main_df["text"]
y = main_df["manipulative"]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

X_multi = main_df["text"]
Y_multi = main_df[MULTILABEL_COLUMNS].astype(int)
X_multi_train, X_multi_test = X_multi.iloc[train_idx], X_multi.iloc[test_idx]
Y_multi_train, Y_multi_test = Y_multi.iloc[train_idx], Y_multi.iloc[test_idx]

print("Training examples:", len(X_train), "| Test examples:", len(X_test))

## Experiment 3A — Word2Vec trained on the project corpus

Word2Vec is trained only on the training texts, not on the held-out test set.

We use:

- 100-dimensional vectors
- Skip-Gram (`sg=1`)
- context window = 5
- negative sampling = 5
- 100 training epochs

The corpus is still small for Word2Vec (a few thousand short texts), so `min_count=1` is used so rare words are not removed.

In [ ]:
# 3A.1 Train Word2Vec for the binary task

w2v_train_tokens = [
    tokenize_text(text)
    for text in X_train
]

w2v_model = Word2Vec(
    sentences=w2v_train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    sg=1,
    negative=5,
    epochs=100,
    seed=RANDOM_STATE
)

print("Vocabulary size:", len(w2v_model.wv))
print("Vector size:", w2v_model.wv.vector_size)

known, total, coverage = embedding_coverage(X_test, w2v_model.wv)
print(f"Test token coverage: {coverage:.1%}")

In [ ]:
# Inspect a few learned semantic relationships

for word in ["only", "left", "hurry", "bought", "ends", "thanks", "stock"]:
    if word in w2v_model.wv:
        print(f"\nMost similar to '{word}':")
        for similar_word, similarity in w2v_model.wv.most_similar(word, topn=5):
            print(f"{similar_word:15s} {similarity:.3f}")

### Experiment 3A.1 — Binary classification with Word2Vec

Each advertisement is represented by the average of its word vectors and classified using Logistic Regression.

In [ ]:
X_train_w2v = vectorize_texts(X_train, w2v_model.wv)
X_test_w2v = vectorize_texts(X_test, w2v_model.wv)

w2v_binary_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

w2v_binary_model.fit(X_train_w2v, y_train)

w2v_binary_pred = w2v_binary_model.predict(X_test_w2v)

w2v_binary_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [
        accuracy_score(y_test, w2v_binary_pred),
        precision_score(y_test, w2v_binary_pred, zero_division=0),
        recall_score(y_test, w2v_binary_pred, zero_division=0),
        f1_score(y_test, w2v_binary_pred, zero_division=0)
    ]
})

display(w2v_binary_metrics.style.format({"Value": "{:.3f}"}))

print(classification_report(
    y_test,
    w2v_binary_pred,
    target_names=["Neutral", "Manipulative"],
    digits=3,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    w2v_binary_pred,
    display_labels=["Neutral", "Manipulative"],
    values_format="d"
)
plt.title("Word2Vec — Binary Classification")
plt.tight_layout()
plt.show()

In [ ]:
# Proper 5-fold CV: Word2Vec is retrained inside every fold

w2v_binary_cv_rows = []

w2v_binary_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (fold_train_idx, fold_val_idx) in enumerate(w2v_binary_cv.split(X, y, groups), start=1):
    fold_X_train = X.iloc[fold_train_idx]
    fold_X_val = X.iloc[fold_val_idx]
    fold_y_train = y.iloc[fold_train_idx]
    fold_y_val = y.iloc[fold_val_idx]

    fold_w2v = Word2Vec(
        sentences=[tokenize_text(text) for text in fold_X_train],
        vector_size=100,
        window=5,
        min_count=1,
        workers=1,
        sg=1,
        negative=5,
        epochs=100,
        seed=RANDOM_STATE
    )

    fold_train_vectors = vectorize_texts(fold_X_train, fold_w2v.wv)
    fold_val_vectors = vectorize_texts(fold_X_val, fold_w2v.wv)

    clf = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    )
    clf.fit(fold_train_vectors, fold_y_train)
    pred = clf.predict(fold_val_vectors)

    w2v_binary_cv_rows.append({
        "fold": fold,
        "accuracy": accuracy_score(fold_y_val, pred),
        "precision": precision_score(fold_y_val, pred, zero_division=0),
        "recall": recall_score(fold_y_val, pred, zero_division=0),
        "f1": f1_score(fold_y_val, pred, zero_division=0)
    })

w2v_binary_cv_results = pd.DataFrame(w2v_binary_cv_rows)
display(w2v_binary_cv_results)

w2v_binary_cv_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        w2v_binary_cv_results["accuracy"].mean(),
        w2v_binary_cv_results["precision"].mean(),
        w2v_binary_cv_results["recall"].mean(),
        w2v_binary_cv_results["f1"].mean()
    ],
    "Std": [
        w2v_binary_cv_results["accuracy"].std(),
        w2v_binary_cv_results["precision"].std(),
        w2v_binary_cv_results["recall"].std(),
        w2v_binary_cv_results["f1"].std()
    ]
})

display(w2v_binary_cv_summary.style.format({
    "Mean": "{:.3f}",
    "Std": "{:.3f}"
}))

### Experiment 3A.2 — Multi-label classification with Word2Vec

A second Word2Vec model is trained only on the multi-label training texts.

The averaged sentence vectors are classified using One-vs-Rest Logistic Regression.

In [ ]:
multi_w2v_model = Word2Vec(
    sentences=[tokenize_text(text) for text in X_multi_train],
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    sg=1,
    negative=5,
    epochs=100,
    seed=RANDOM_STATE
)

X_multi_train_w2v = vectorize_texts(X_multi_train, multi_w2v_model.wv)
X_multi_test_w2v = vectorize_texts(X_multi_test, multi_w2v_model.wv)

w2v_multilabel_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    )
)

w2v_multilabel_model.fit(X_multi_train_w2v, Y_multi_train)
w2v_multi_pred = w2v_multilabel_model.predict(X_multi_test_w2v)

w2v_multi_metrics = pd.DataFrame({
    "Metric": [
        "Subset accuracy",
        "Hamming loss",
        "Micro Precision",
        "Micro Recall",
        "Micro F1",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Micro Jaccard"
    ],
    "Value": [
        accuracy_score(Y_multi_test, w2v_multi_pred),
        hamming_loss(Y_multi_test, w2v_multi_pred),
        precision_score(Y_multi_test, w2v_multi_pred, average="micro", zero_division=0),
        recall_score(Y_multi_test, w2v_multi_pred, average="micro", zero_division=0),
        f1_score(Y_multi_test, w2v_multi_pred, average="micro", zero_division=0),
        precision_score(Y_multi_test, w2v_multi_pred, average="macro", zero_division=0),
        recall_score(Y_multi_test, w2v_multi_pred, average="macro", zero_division=0),
        f1_score(Y_multi_test, w2v_multi_pred, average="macro", zero_division=0),
        jaccard_score(Y_multi_test, w2v_multi_pred, average="micro", zero_division=0)
    ]
})

display(w2v_multi_metrics.style.format({"Value": "{:.3f}"}))

In [ ]:
# Multi-label performance by technique

w2v_per_label_rows = []

for i, label in enumerate(MULTILABEL_COLUMNS):
    w2v_per_label_rows.append({
        "label": label,
        "precision": precision_score(
            Y_multi_test.iloc[:, i],
            w2v_multi_pred[:, i],
            zero_division=0
        ),
        "recall": recall_score(
            Y_multi_test.iloc[:, i],
            w2v_multi_pred[:, i],
            zero_division=0
        ),
        "f1": f1_score(
            Y_multi_test.iloc[:, i],
            w2v_multi_pred[:, i],
            zero_division=0
        ),
        "support": int(Y_multi_test.iloc[:, i].sum())
    })

w2v_per_label_metrics = pd.DataFrame(w2v_per_label_rows).sort_values(
    "f1",
    ascending=False
)

display(w2v_per_label_metrics.style.format({
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}"
}))

In [ ]:
# Proper 5-fold CV for multi-label Word2Vec

w2v_multi_cv_rows = []

w2v_multi_cv = GroupKFold(n_splits=5)

for fold, (fold_train_idx, fold_val_idx) in enumerate(w2v_multi_cv.split(X_multi, groups=groups), start=1):
    fold_X_train = X_multi.iloc[fold_train_idx]
    fold_X_val = X_multi.iloc[fold_val_idx]
    fold_Y_train = Y_multi.iloc[fold_train_idx]
    fold_Y_val = Y_multi.iloc[fold_val_idx]

    fold_w2v = Word2Vec(
        sentences=[tokenize_text(text) for text in fold_X_train],
        vector_size=100,
        window=5,
        min_count=1,
        workers=1,
        sg=1,
        negative=5,
        epochs=100,
        seed=RANDOM_STATE
    )

    fold_train_vectors = vectorize_texts(fold_X_train, fold_w2v.wv)
    fold_val_vectors = vectorize_texts(fold_X_val, fold_w2v.wv)

    clf = OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        )
    )
    clf.fit(fold_train_vectors, fold_Y_train)
    pred = clf.predict(fold_val_vectors)

    w2v_multi_cv_rows.append({
        "fold": fold,
        "micro_f1": f1_score(
            fold_Y_val,
            pred,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            fold_Y_val,
            pred,
            average="macro",
            zero_division=0
        )
    })

w2v_multi_cv_results = pd.DataFrame(w2v_multi_cv_rows)
display(w2v_multi_cv_results)

w2v_multi_cv_summary = pd.DataFrame({
    "Metric": ["Micro F1", "Macro F1"],
    "Mean": [
        w2v_multi_cv_results["micro_f1"].mean(),
        w2v_multi_cv_results["macro_f1"].mean()
    ],
    "Std": [
        w2v_multi_cv_results["micro_f1"].std(),
        w2v_multi_cv_results["macro_f1"].std()
    ]
})

display(w2v_multi_cv_summary.style.format({
    "Mean": "{:.3f}",
    "Std": "{:.3f}"
}))

## Experiment 3B — Pretrained GloVe embeddings

The Word2Vec model above learns only from the small project corpus.

Now we use `glove-wiki-gigaword-100`, a pretrained 100-dimensional embedding model learned from a much larger external corpus.

The first run downloads the model, so internet access is required.

In [ ]:
# Download/load pretrained GloVe embeddings

glove_model = api.load("glove-wiki-gigaword-100")

print("GloVe vocabulary size:", len(glove_model))
print("GloVe vector size:", glove_model.vector_size)

known, total, coverage = embedding_coverage(X_test, glove_model)
print(f"GloVe test token coverage: {coverage:.1%}")

In [ ]:
# Inspect a few semantic relationships in pretrained GloVe

for word in ["buy", "risk", "family", "expert", "urgent"]:
    if word in glove_model:
        print(f"\nMost similar to '{word}':")
        for similar_word, similarity in glove_model.most_similar(word, topn=5):
            print(f"{similar_word:15s} {similarity:.3f}")

### Experiment 3B.1 — Binary classification with pretrained GloVe

In [ ]:
X_train_glove = vectorize_texts(X_train, glove_model)
X_test_glove = vectorize_texts(X_test, glove_model)

glove_binary_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

glove_binary_model.fit(X_train_glove, y_train)
glove_binary_pred = glove_binary_model.predict(X_test_glove)

glove_binary_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [
        accuracy_score(y_test, glove_binary_pred),
        precision_score(y_test, glove_binary_pred, zero_division=0),
        recall_score(y_test, glove_binary_pred, zero_division=0),
        f1_score(y_test, glove_binary_pred, zero_division=0)
    ]
})

display(glove_binary_metrics.style.format({"Value": "{:.3f}"}))

### Experiment 3B.2 — Multi-label classification with pretrained GloVe

In [ ]:
X_multi_train_glove = vectorize_texts(X_multi_train, glove_model)
X_multi_test_glove = vectorize_texts(X_multi_test, glove_model)

glove_multilabel_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    )
)

glove_multilabel_model.fit(X_multi_train_glove, Y_multi_train)
glove_multi_pred = glove_multilabel_model.predict(X_multi_test_glove)

glove_multi_metrics = pd.DataFrame({
    "Metric": [
        "Subset accuracy",
        "Hamming loss",
        "Micro Precision",
        "Micro Recall",
        "Micro F1",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Micro Jaccard"
    ],
    "Value": [
        accuracy_score(Y_multi_test, glove_multi_pred),
        hamming_loss(Y_multi_test, glove_multi_pred),
        precision_score(Y_multi_test, glove_multi_pred, average="micro", zero_division=0),
        recall_score(Y_multi_test, glove_multi_pred, average="micro", zero_division=0),
        f1_score(Y_multi_test, glove_multi_pred, average="micro", zero_division=0),
        precision_score(Y_multi_test, glove_multi_pred, average="macro", zero_division=0),
        recall_score(Y_multi_test, glove_multi_pred, average="macro", zero_division=0),
        f1_score(Y_multi_test, glove_multi_pred, average="macro", zero_division=0),
        jaccard_score(Y_multi_test, glove_multi_pred, average="micro", zero_division=0)
    ]
})

display(glove_multi_metrics.style.format({"Value": "{:.3f}"}))

In [ ]:
# Cross-validation for pretrained GloVe
# The embeddings are fixed external features, so only the classifiers are retrained per fold.

X_glove_all = vectorize_texts(X, glove_model)

glove_binary_cv_rows = []
glove_binary_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (fold_train_idx, fold_val_idx) in enumerate(
    glove_binary_cv.split(X_glove_all, y, groups),
    start=1
):
    clf = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    )
    clf.fit(X_glove_all[fold_train_idx], y.iloc[fold_train_idx])
    pred = clf.predict(X_glove_all[fold_val_idx])

    glove_binary_cv_rows.append({
        "fold": fold,
        "accuracy": accuracy_score(y.iloc[fold_val_idx], pred),
        "precision": precision_score(y.iloc[fold_val_idx], pred, zero_division=0),
        "recall": recall_score(y.iloc[fold_val_idx], pred, zero_division=0),
        "f1": f1_score(y.iloc[fold_val_idx], pred, zero_division=0)
    })

glove_binary_cv_results = pd.DataFrame(glove_binary_cv_rows)

glove_binary_cv_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        glove_binary_cv_results["accuracy"].mean(),
        glove_binary_cv_results["precision"].mean(),
        glove_binary_cv_results["recall"].mean(),
        glove_binary_cv_results["f1"].mean()
    ],
    "Std": [
        glove_binary_cv_results["accuracy"].std(),
        glove_binary_cv_results["precision"].std(),
        glove_binary_cv_results["recall"].std(),
        glove_binary_cv_results["f1"].std()
    ]
})

display(glove_binary_cv_summary.style.format({
    "Mean": "{:.3f}",
    "Std": "{:.3f}"
}))

X_multi_glove_all = vectorize_texts(X_multi, glove_model)

glove_multi_cv_rows = []
glove_multi_cv = GroupKFold(n_splits=5)

for fold, (fold_train_idx, fold_val_idx) in enumerate(
    glove_multi_cv.split(X_multi_glove_all, groups=groups),
    start=1
):
    clf = OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        )
    )

    clf.fit(
        X_multi_glove_all[fold_train_idx],
        Y_multi.iloc[fold_train_idx]
    )

    pred = clf.predict(
        X_multi_glove_all[fold_val_idx]
    )

    glove_multi_cv_rows.append({
        "fold": fold,
        "micro_f1": f1_score(
            Y_multi.iloc[fold_val_idx],
            pred,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            Y_multi.iloc[fold_val_idx],
            pred,
            average="macro",
            zero_division=0
        )
    })

glove_multi_cv_results = pd.DataFrame(glove_multi_cv_rows)

glove_multi_cv_summary = pd.DataFrame({
    "Metric": ["Micro F1", "Macro F1"],
    "Mean": [
        glove_multi_cv_results["micro_f1"].mean(),
        glove_multi_cv_results["macro_f1"].mean()
    ],
    "Std": [
        glove_multi_cv_results["micro_f1"].std(),
        glove_multi_cv_results["macro_f1"].std()
    ]
})

display(glove_multi_cv_summary.style.format({
    "Mean": "{:.3f}",
    "Std": "{:.3f}"
}))

## Experiment 3 — Final comparison

The table below compares the three text representations on both tasks.

The important question is not only which model obtains the highest score, but also whether dense semantic information improves generalization compared with the lexical TF-IDF baseline.

In [ ]:
# TF-IDF results from Experiments 1 and 2

metrics = load_table("exp1_test_metrics")
cv_summary = load_table("exp1_cv_summary")
multi_metrics = load_table("exp2_test_metrics")
multi_cv_summary = load_table("exp2_cv_summary")

In [ ]:
def get_value(table, metric):
    return table.loc[
        table["Metric"] == metric,
        "Value"
    ].iloc[0]

def get_mean(table, metric):
    return table.loc[
        table["Metric"] == metric,
        "Mean"
    ].iloc[0]

comparison_test = pd.DataFrame([
    {
        "Representation": "TF-IDF",
        "Binary F1": get_value(metrics, "F1-score"),
        "Multi-label Micro F1": get_value(multi_metrics, "Micro F1"),
        "Multi-label Macro F1": get_value(multi_metrics, "Macro F1")
    },
    {
        "Representation": "Word2Vec — project corpus",
        "Binary F1": get_value(w2v_binary_metrics, "F1-score"),
        "Multi-label Micro F1": get_value(w2v_multi_metrics, "Micro F1"),
        "Multi-label Macro F1": get_value(w2v_multi_metrics, "Macro F1")
    },
    {
        "Representation": "Pretrained GloVe",
        "Binary F1": get_value(glove_binary_metrics, "F1-score"),
        "Multi-label Micro F1": get_value(glove_multi_metrics, "Micro F1"),
        "Multi-label Macro F1": get_value(glove_multi_metrics, "Macro F1")
    }
])

print("Held-out test comparison:")
display(comparison_test.style.format({
    "Binary F1": "{:.3f}",
    "Multi-label Micro F1": "{:.3f}",
    "Multi-label Macro F1": "{:.3f}"
}))

comparison_cv = pd.DataFrame([
    {
        "Representation": "TF-IDF",
        "Binary CV F1": get_mean(cv_summary, "F1-score"),
        "Multi-label CV Micro F1": get_mean(multi_cv_summary, "Micro F1"),
        "Multi-label CV Macro F1": get_mean(multi_cv_summary, "Macro F1")
    },
    {
        "Representation": "Word2Vec — project corpus",
        "Binary CV F1": get_mean(w2v_binary_cv_summary, "F1-score"),
        "Multi-label CV Micro F1": get_mean(w2v_multi_cv_summary, "Micro F1"),
        "Multi-label CV Macro F1": get_mean(w2v_multi_cv_summary, "Macro F1")
    },
    {
        "Representation": "Pretrained GloVe",
        "Binary CV F1": get_mean(glove_binary_cv_summary, "F1-score"),
        "Multi-label CV Micro F1": get_mean(glove_multi_cv_summary, "Micro F1"),
        "Multi-label CV Macro F1": get_mean(glove_multi_cv_summary, "Macro F1")
    }
])

print("Cross-validation comparison:")
display(comparison_cv.style.format({
    "Binary CV F1": "{:.3f}",
    "Multi-label CV Micro F1": "{:.3f}",
    "Multi-label CV Macro F1": "{:.3f}"
}))

## Experiment 3 — External comparison on our original data

Internal scores on the main dataset are close to the ceiling for all methods, so the more informative question is **which representation transfers better** to our own advertisements.
Each representation is trained on the full main dataset and evaluated (binary F1) on our real and synthetic examples.

In [ ]:
# The TF-IDF baseline from Experiment 1, retrained on the full main dataset

final_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
final_baseline.fit(main_df["text"], main_df["manipulative"])

In [ ]:
# Train each representation on the full main dataset, evaluate on the external sets

full_w2v = Word2Vec(
    sentences=[tokenize_text(t) for t in main_df["text"]],
    vector_size=100, window=5, min_count=1, workers=1,
    sg=1, negative=5, epochs=100, seed=RANDOM_STATE
)

def fit_dense(kv):
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(vectorize_texts(main_df["text"], kv), main_df["manipulative"])
    return lambda texts: clf.predict(vectorize_texts(texts, kv))

predictors = {
    "TF-IDF": final_baseline.predict,
    "Word2Vec — project corpus": fit_dense(full_w2v.wv),
    "Pretrained GloVe": fit_dense(glove_model),
}

external_sets = {
    "Real ads": real_df,
    "Synthetic (all)": external_synthetic_df,
    "Synthetic hard negatives": external_synthetic_df[external_synthetic_df["source"] == "synthetic_hard_negative"],
}

rows = []
for name, predict in predictors.items():
    row = {"Representation": name}
    for set_name, data in external_sets.items():
        pred = predict(data["text"])
        if data["manipulative"].nunique() > 1:
            row[f"{set_name} F1"] = f1_score(data["manipulative"], pred, zero_division=0)
        else:
            row[f"{set_name} accuracy"] = accuracy_score(data["manipulative"], pred)
    rows.append(row)

comparison_external = pd.DataFrame(rows)
print("External comparison (trained on the full main dataset):")
display(comparison_external.style.format({c: "{:.3f}" for c in comparison_external.columns if c != "Representation"}))

In [ ]:
# Save results for the comparison in Experiment 4

save_table(glove_binary_metrics, "exp3_glove_binary_metrics")
save_table(comparison_test, "exp3_comparison_test")
save_table(comparison_cv, "exp3_comparison_cv")
save_table(comparison_external, "exp3_comparison_external")
print("Saved to results/.")

## Experiment 3 — Interpretation

Experiment 3 compares three levels of text representation:

1. **TF-IDF:** sparse lexical information.
2. **Project Word2Vec:** dense embeddings learned only from the training texts.
3. **Pretrained GloVe:** dense embeddings learned from a much larger external corpus.

Two views matter:
- **Internal scores** (held-out test and cross-validation on the dark patterns data) show how well each representation fits this domain.
- **External scores** (our real and synthetic examples) show how well each representation transfers to a different style of advertising text. The hard-negative accuracy is especially informative: it shows whether a model reacts to keywords or to the actual manipulative framing.

## Results summary (from this run)

| | TF-IDF | Word2Vec (project) | GloVe |
|---|---|---|---|
| Binary F1 — held-out test | 0.932 | **0.940** | 0.904 |
| Binary F1 — 5-fold group CV | **0.946** | 0.926 | 0.896 |
| Technique Macro F1 — 5-fold group CV | **0.943** | 0.880 | 0.810 |
| Binary F1 — our real ads (n=57) | 0.468 | **0.571** | 0.415 |
| Binary F1 — our synthetic set (n=370) | 0.414 | **0.498** | 0.404 |
| Accuracy — our hard negatives (n=90) | **0.822** | 0.778 | 0.800 |

**Inside the dark patterns dataset**, TF-IDF is the strongest or tied: dark patterns on websites are highly templated (*"Only N left"*, *"X just bought"*, *"No thanks, I don't…"*), which suits a lexical model. Word2Vec's single-split win (0.940 vs 0.932) is within cross-validation noise; over 5 folds TF-IDF is ahead. For technique classification the gap is clearer.

**On our own data**, Word2Vec transfers best: it beats TF-IDF by about 0.10 F1 on both external sets. Dense vectors let the model match words it never saw in training (e.g. *"remain"* close to *"left"*), which TF-IDF cannot do. The real set has only 57 examples, though, so this difference corresponds to a handful of texts and should be confirmed on a larger real set.

**All models drop sharply on our data** (≈0.95 → ≈0.4–0.6 F1). The breakdown in section 9b explains why: the model finds 82–90% of neutral texts correctly but recognizes only ~30% of our manipulative ad copy. The dark patterns data teaches scarcity, urgency, social proof, and confirmshaming in website-UI form; our ads also use fear, exaggeration, and authority, which never appear in training. Hard negatives with words like *"ends"* are still sometimes flagged, showing remaining keyword reliance.

**Takeaways for the project**
1. The new dataset is large enough that the Word2Vec vs TF-IDF comparison is now meaningful, and the answer depends on the evaluation: TF-IDF wins in-domain, Word2Vec generalizes better out-of-domain.
2. To detect fear, exaggeration, and authority, we still need labeled examples of them — e.g. from the SemEval persuasion-technique datasets or more real ads.
3. Expanding the real-ad test set is the most valuable next step for making the external comparison reliable.